> ⚠️ **Methodisch überholt (2026-07-31), noch am selben Tag ersetzt:** Diese Analyse nutzt die statischen Gruppen `female_top`/`male_control` (`players.analysis_group`), die sich beim Testlauf dieses Notebooks als nur unvollständig befüllt herausstellten (`groups.backfill_status='partial'`, nur 23/66 bzw. 48/649 Spieler gelabelt, davon kaum aktive). Die Tabellen unten liefen fehlerfrei, aber mit sehr dünnen Zellen (nur 856 Partien). Für den Frauen-vs-Männer-Vergleich siehe stattdessen **Notebook 14** (`14_top50_female_vs_band_men.ipynb`): survivorship-bias-freie Kohorte (Top-50-Frauen nach `rating_history.published_rating` je Jahresende 2021–2025, 70 Spielerinnen) vs. Männer im selben Elo-Band, dynamisch aus `rating_history` abgeleitet — 14.005 bzw. 88.188 bereits gescrapte Partien, kein neues Scraping nötig (siehe `docs/ideen_verbesserungen.md`, Abschnitte F2/F7). Notebook 14 übernimmt die hier entwickelte Metrik-/Chart-/Signifikanztest-Logik unverändert, nur mit der neuen Kohorten-Definition.

# 13 — Absoluter Elo-Vergleich: female_top vs. male_control

**Kernfrage:** Ist eine Frau mit z.B. Elo 2500 gegen einen gleich starken Gegner leichter, stärker oder schwächer als ein Mann mit Elo 2500?

Notebook 07 beantwortet eine verwandte, aber andere Frage: wie schneiden `female_top`-Spielerinnen (für sich allein) gegen *relativ* stärkere/schwächere Gegner ab. Hier vergleichen wir stattdessen **beide Gruppen direkt**, gebündelt nach dem **absoluten** eigenen Rating (nicht nur der Differenz zum Gegner) — `female_top` und `male_control` sind als parallele, alters-gematchte Kohorten mit identischer Elo-Range (2400–2600) angelegt und damit direkt vergleichbar.

**Definitionen:**
- **Elo-Band (Primärachse):** eigenes Rating in 50-Punkte-Bändern (`2400-2449`, `2450-2499`, …). Werte außerhalb 2400–2600 entstehen durch normale Rating-Schwankung über die Zeit und werden als eigene Bänder mitgeführt, nicht gefiltert.
- **Stärke-Bucket (Sekundärachse, relativ):** `gleich` = |Gegner − ich| ≤ 50, `stärker`/`schwächer` sonst. **Schwelle ±50** — abweichend von Notebook 07 (±80). Notebook 07 bleibt unverändert (historisch); ±50 (wie in Notebook 05) wird hiermit als Projekt-Standard für neue Notebooks festgelegt.
- **Scope:** `tournament_type ∈ {women, women_team}` → `women_only`, sonst `open_mixed`. `female_top` bestreitet laut `docs/datenuebersicht.md` 65,4 % ihrer Partien gegen Frauen, überwiegend in Frauen-only-Turnieren. Wir filtern das **nicht heraus**, zeigen aber jede Kerntabelle zusätzlich für `open_mixed` allein, damit der Effekt sichtbar bleibt statt versteckt zu werden.

**Filter:** `active = TRUE`, `analysis_group IN ('female_top','male_control')`, `opponent_sex IN ('M','F')`, eigenes Rating vorhanden.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _setup import load_query, apply_style, GROUP_PALETTE, GROUP_ORDER

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

apply_style()
pd.set_option('display.max_rows', 250)

## Datenbasis laden

In [ ]:
sql = '''
SELECT
    gr.fide_id,
    p.analysis_group,
    rh.std_rating          AS own_rating,
    gr.opponent_rating,
    gr.opponent_sex,
    gr.result,
    gr.rating_change_weighted,
    gr.expected_score,
    gr.over_performance,
    gr.tournament_type
FROM game_results gr
JOIN players p               ON p.fide_id = gr.fide_id
LEFT JOIN rating_history rh  ON rh.fide_id = gr.fide_id AND rh.period = gr.period
WHERE p.active = TRUE
  AND p.analysis_group IN ('female_top', 'male_control')
  AND gr.opponent_sex IN ('M', 'F')
'''
raw = load_query(sql)
n_raw = len(raw)
df = raw.dropna(subset=['own_rating', 'opponent_rating']).copy()
print(f'{n_raw:,} Partien geladen, {n_raw - len(df):,} ohne own_rating/opponent_rating verworfen '
      f'({(n_raw - len(df)) / n_raw:.1%}) -> {len(df):,} Partien in der Analyse')

df['result'] = df['result'].astype(float)
df['rating_change_weighted'] = df['rating_change_weighted'].astype(float)
df['expected_score'] = df['expected_score'].astype(float)
df['over_performance'] = df['over_performance'].astype(float)
df['own_rating'] = pd.to_numeric(df['own_rating'], errors='coerce')
df['opponent_rating'] = pd.to_numeric(df['opponent_rating'], errors='coerce')
df['diff'] = df['opponent_rating'] - df['own_rating']

def elo_band(r):
    if pd.isna(r):
        return 'unknown'
    lo = int(r // 50) * 50
    return f'{lo}-{lo + 49}'
df['elo_band'] = df['own_rating'].apply(elo_band)

def strength_bucket(d):
    if pd.isna(d):
        return 'unknown'
    if d > 50:
        return 'stärker'
    if d < -50:
        return 'schwächer'
    return 'gleich'
df['strength'] = df['diff'].apply(strength_bucket)

df['scope'] = df['tournament_type'].apply(
    lambda t: 'women_only' if t in ('women', 'women_team') else 'open_mixed'
)

def band_sort_key(b):
    return (9999,) if b == 'unknown' else (int(b.split('-')[0]),)
elo_band_order = sorted(df['elo_band'].unique(), key=band_sort_key)

SEX_ORDER = ['F', 'M']
STRENGTH_ORDER = ['stärker', 'gleich', 'schwächer']
print('Elo-Bänder:', elo_band_order)
df.head()

## 1. Zellgrößen (Plausibilitätscheck)

`female_top` hat nur 66 Spielerinnen — an den Rändern der Elo-Range können einzelne Bänder sehr dünn besetzt sein. Vor jeder weiteren Aufschlüsselung prüfen.

In [ ]:
qc = (
    df.groupby(['analysis_group', 'elo_band'])
      .agg(n_games=('fide_id', 'size'), n_players=('fide_id', 'nunique'))
      .reset_index()
)
qc.pivot(index='elo_band', columns='analysis_group', values=['n_games', 'n_players']).reindex(elo_band_order)

## Helper: Metrik-Tabellen

In [ ]:
def build_metrics(df_scope):
    g = df_scope.groupby(['elo_band', 'opponent_sex', 'analysis_group'])
    return g.agg(
        n_games=('fide_id', 'size'),
        n_players=('fide_id', 'nunique'),
        score_rate=('result', 'mean'),
        mean_expected_score=('expected_score', 'mean'),
        mean_over_performance=('over_performance', 'mean'),
        sum_rating_change_weighted=('rating_change_weighted', 'sum'),
        mean_rating_change_weighted=('rating_change_weighted', 'mean'),
    ).reset_index()

def pivot_metric(metrics_df, value_col, round_to=4):
    tbl = metrics_df.pivot(index='elo_band', columns=['opponent_sex', 'analysis_group'], values=value_col)
    tbl = tbl.reindex(elo_band_order)
    cols = [(s, grp) for s in SEX_ORDER for grp in GROUP_ORDER if (s, grp) in tbl.columns]
    return tbl[cols].round(round_to)

metrics_all = build_metrics(df)
metrics_open = build_metrics(df[df.scope == 'open_mixed'])
print(f'Zellen gesamt: {len(metrics_all)}   Zellen (nur offene Turniere): {len(metrics_open)}')

## 2. Haupttabelle — Ø Over-Performance je Elo-Band × Gegner-Geschlecht × Gruppe

`over_performance = result − expected_score` (Elo-Erwartung). Positiv = besser als erwartet.

**Alle Partien:**

In [ ]:
pivot_metric(metrics_all, 'mean_over_performance')

**Nur offene/gemischte Turniere** (ohne `women`/`women_team`):

In [ ]:
pivot_metric(metrics_open, 'mean_over_performance')

### Zum Vergleich: Score-Rate und Ø rating_change_weighted (alle Partien)

In [ ]:
pivot_metric(metrics_all, 'score_rate')

In [ ]:
pivot_metric(metrics_all, 'mean_rating_change_weighted')

## 3. Heatmap: Ø Over-Performance nach Elo-Band × Gegner-Geschlecht

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, grp in zip(axes, GROUP_ORDER):
    sub = (
        metrics_all[metrics_all.analysis_group == grp]
        .pivot(index='elo_band', columns='opponent_sex', values='mean_over_performance')
        .reindex(index=elo_band_order, columns=SEX_ORDER)
    )
    sns.heatmap(sub, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=ax, cbar=True)
    ax.set_title(grp)
    ax.set_xlabel('Gegner-Geschlecht')
    ax.set_ylabel('Elo-Band (eigenes Rating)')
plt.tight_layout(); plt.show()

## 4. Balkendiagramm: Ø rating_change_weighted je Elo-Band, Gruppe im Vergleich

In [ ]:
def bar_chart(metrics_df, title_suffix):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    for ax, sex in zip(axes, SEX_ORDER):
        sub = metrics_df[metrics_df.opponent_sex == sex]
        pv = (
            sub.pivot(index='elo_band', columns='analysis_group', values='mean_rating_change_weighted')
            .reindex(index=elo_band_order, columns=GROUP_ORDER)
        )
        pv.plot.bar(ax=ax, color=[GROUP_PALETTE[g] for g in GROUP_ORDER], edgecolor='white')
        ax.axhline(0, color='grey', lw=0.8, ls='--')
        ax.set_title(f'vs {sex} ({title_suffix})')
        ax.set_xlabel('Elo-Band (eigenes Rating)')
        ax.set_ylabel('Ø rating_change_weighted')
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.show()

bar_chart(metrics_all, 'alle Partien')

In [ ]:
bar_chart(metrics_open, 'nur offene Turniere')

## 5. Sekundärachse: relative Gegnerstärke (±50 Elo)

Zur Einordnung neben der absoluten Elo-Band-Sicht — analog zu Notebook 05, aber hier direkt `female_top` vs. `male_control` gegenübergestellt.

In [ ]:
def build_strength_metrics(df_scope):
    g = df_scope.groupby(['strength', 'opponent_sex', 'analysis_group'])
    return g.agg(
        n_games=('fide_id', 'size'),
        n_players=('fide_id', 'nunique'),
        score_rate=('result', 'mean'),
        mean_over_performance=('over_performance', 'mean'),
        mean_rating_change_weighted=('rating_change_weighted', 'mean'),
    ).reset_index()

strength_metrics = build_strength_metrics(df)
tbl = strength_metrics.pivot(index='strength', columns=['opponent_sex', 'analysis_group'], values='mean_over_performance')
tbl = tbl.reindex(STRENGTH_ORDER)
cols = [(s, grp) for s in SEX_ORDER for grp in GROUP_ORDER if (s, grp) in tbl.columns]
tbl[cols].round(4)

## 6. Signifikanztest

**Problem:** Partien sind pro Spieler geclustert — `female_top` hat 66 Spielerinnen, `male_control` 649 Spieler, mit sehr ungleicher Partienzahl pro Person. Ein Test auf Partie-Ebene würde Pseudo-Replikation erzeugen und vielspielende Personen überproportional gewichten.

**Ansatz:** Zweistufig, nur mit `numpy` (kein `scipy`/`statsmodels` im Projekt installiert, bewusst keine neue Abhängigkeit für einen einzelnen Test):
1. Pro Zelle (Elo-Band × Gegner-Geschlecht × Scope) erst auf **Spieler-Ebene** aggregieren (Ø `over_performance` pro `fide_id`) — macht aus korrelierten Partien unabhängige Beobachtungen.
2. Zweiseitiger **Permutationstest** auf den Mittelwertsunterschied der Spieler-Mittelwerte (10.000 Permutationen, fester Seed). Macht keine Normalverteilungsannahme — wichtig, da `over_performance` pro Spieler bei kleinem `n_games` stark schief verteilt sein kann.

Zellen mit weniger als 8 Spielern in einer der beiden Gruppen werden als `underpowered` markiert statt unterdrückt — `female_top` dünnt an den Rändern der Elo-Range schnell aus.

In [ ]:
def permutation_test(a, b, n_perm=10000, seed=42, return_diffs=False):
    rng = np.random.default_rng(seed)
    observed = a.mean() - b.mean()
    pooled = np.concatenate([a, b])
    n_a = len(a)
    diffs = np.empty(n_perm)
    for i in range(n_perm):
        rng.shuffle(pooled)
        diffs[i] = pooled[:n_a].mean() - pooled[n_a:].mean()
    p_value = (np.abs(diffs) >= np.abs(observed)).mean()
    return (observed, p_value, diffs) if return_diffs else (observed, p_value)

def player_level_means(df_scope, value_col):
    return df_scope.groupby(['analysis_group', 'fide_id'])[value_col].mean().reset_index()

SCOPES = {'all': df, 'open_mixed': df[df.scope == 'open_mixed']}
sig_rows = []
for scope_name, dsub in SCOPES.items():
    for band in elo_band_order:
        if band == 'unknown':
            continue
        for sex in SEX_ORDER:
            cell = dsub[(dsub.elo_band == band) & (dsub.opponent_sex == sex)]
            pm = player_level_means(cell, 'over_performance')
            a = pm.loc[pm.analysis_group == 'female_top', 'over_performance'].values
            b = pm.loc[pm.analysis_group == 'male_control', 'over_performance'].values
            if len(a) == 0 or len(b) == 0:
                continue
            diff, p = permutation_test(a, b)
            sig_rows.append({
                'scope': scope_name, 'elo_band': band, 'opponent_sex': sex,
                'n_players_female_top': len(a), 'n_players_male_control': len(b),
                'mean_diff': round(diff, 4), 'p_value': round(p, 4),
                'underpowered': len(a) < 8 or len(b) < 8,
            })
sig_cols = ['scope', 'elo_band', 'opponent_sex', 'n_players_female_top',
            'n_players_male_control', 'mean_diff', 'p_value', 'underpowered']
sig_table = pd.DataFrame(sig_rows, columns=sig_cols)
if sig_table.empty:
    print('Keine Zelle hat aktuell Spieler in beiden Gruppen (female_top UND male_control) — '
          'vermutlich weil der Backfill für eine der beiden Kohorten noch läuft '
          '(siehe groups.backfill_status). Sobald mehr Partien vorliegen, füllt sich diese Tabelle.')
sig_table

### Zur Veranschaulichung: Nullverteilung der am besten besetzten Zelle

In [ ]:
all_scope = sig_table[sig_table.scope == 'all']
if all_scope.empty:
    print('Übersprungen: keine Zelle mit Daten in beiden Gruppen (siehe Hinweis oben).')
else:
    best = all_scope.sort_values(
        ['n_players_female_top', 'n_players_male_control'], ascending=False
    ).iloc[0]
    cell = df[(df.elo_band == best.elo_band) & (df.opponent_sex == best.opponent_sex)]
    pm = player_level_means(cell, 'over_performance')
    a = pm.loc[pm.analysis_group == 'female_top', 'over_performance'].values
    b = pm.loc[pm.analysis_group == 'male_control', 'over_performance'].values
    observed, p_value, diffs = permutation_test(a, b, return_diffs=True)

    fig, ax = plt.subplots()
    ax.hist(diffs, bins=50, color='#888888', alpha=0.8)
    ax.axvline(observed, color=GROUP_PALETTE['female_top'], lw=2,
               label=f'beobachtet ({observed:+.3f}), p={p_value:.4f}')
    ax.set_title(f'Permutations-Nullverteilung: {best.elo_band}, vs {best.opponent_sex}')
    ax.set_xlabel('Differenz der Spieler-Mittelwerte (female_top − male_control)')
    ax.legend()
    plt.tight_layout(); plt.show()

## Fazit

Die Tabellen und der Signifikanztest oben beantworten die Ausgangsfrage direkt: pro Elo-Band zeigt die `mean_over_performance`-Tabelle (Abschnitt 2), ob `female_top`- oder `male_control`-Spieler:innen bei gleichem absoluten Rating besser oder schlechter abschneiden als die Elo-Erwartung vorgibt — getrennt nach Gegner-Geschlecht und mit/ohne Frauen-only-Turniere. Der Permutationstest (Abschnitt 6) zeigt, welche dieser Unterschiede über die reine Stichprobenstreuung hinausgehen.

**Caveats:**
- `female_top` hat nur 66 Spielerinnen — Zellen an den Rändern der Elo-Range (sehr niedrige/hohe Bänder) sind oft `underpowered` und sollten nicht überinterpretiert werden.
- Der Frauen-Turnier-Bias (65,4 % der `female_top`-Partien gegen Frauen, meist Frauen-only) bleibt in der `all`-Sicht enthalten; die `open_mixed`-Sicht ist die fairere Vergleichsbasis, hat aber pro Zelle weniger Partien/Spieler:innen.